# Exam (Easy) — Practice Problems

Confidence builders. Each problem is 15-20 min. If you can't solve these under stress, focus here before moving to MEDIUM.

**Mix:** 50% data pipelines / 50% classic ML basics

**Rules:**
- Set a timer per problem
- No docs, no autocomplete, no outside help
- Run the test cell — all assertions must pass

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from typing import Any
from collections import Counter
import math
import random

---

## Problem 1: Channel-wise Image Statistics

**Scenario:** You're building a data ingestion pipeline for a video dataset. Before feeding frames into a model, you need to compute per-channel normalization statistics across a batch of images.

**Spec:**

Given a batch of images as a tensor of shape `(B, C, H, W)`, compute per-channel statistics across the entire batch (i.e., aggregate over B, H, W dimensions).

Return a dict with keys:
- `"mean"` — tensor of shape `(C,)`, per-channel mean
- `"std"` — tensor of shape `(C,)`, per-channel standard deviation (unbiased, i.e. default `torch.std`)
- `"min"` — tensor of shape `(C,)`, per-channel minimum
- `"max"` — tensor of shape `(C,)`, per-channel maximum

**Constraints:**
- No loops. Use tensor operations only.
- Return float32 tensors regardless of input dtype.

In [ ]:
def channel_statistics(images: torch.Tensor) -> dict[str, torch.Tensor]:
    """Compute per-channel mean, std, min, max for a batch of images.

    Args:
        images: (B, C, H, W) tensor

    Returns:
        Dict with keys 'mean', 'std', 'min', 'max', each a (C,) tensor.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 1 ---
torch.manual_seed(42)
imgs = torch.randn(8, 3, 32, 32)
stats = channel_statistics(imgs)

# Check keys
assert set(stats.keys()) == {"mean", "std", "min", "max"}

# Check shapes
for key in stats:
    assert stats[key].shape == (3,), f"{key} shape should be (3,), got {stats[key].shape}"

# Check dtypes
for key in stats:
    assert stats[key].dtype == torch.float32, f"{key} should be float32"

# Check values against reference
for c in range(3):
    channel_data = imgs[:, c, :, :]
    assert torch.allclose(stats["mean"][c], channel_data.mean(), atol=1e-5)
    assert torch.allclose(stats["std"][c], channel_data.std(), atol=1e-5)
    assert torch.allclose(stats["min"][c], channel_data.min(), atol=1e-5)
    assert torch.allclose(stats["max"][c], channel_data.max(), atol=1e-5)

# Edge case: single image
single = torch.tensor([[[[1.0, 2.0], [3.0, 4.0]], [[5.0, 6.0], [7.0, 8.0]]]])  # (1,2,2,2)
s = channel_statistics(single)
assert s["mean"].shape == (2,)
assert torch.allclose(s["mean"][0], torch.tensor(2.5))
assert torch.allclose(s["min"][1], torch.tensor(5.0))
assert torch.allclose(s["max"][1], torch.tensor(8.0))

# Edge case: integer input should return float32
int_imgs = torch.randint(0, 256, (4, 3, 8, 8))
s2 = channel_statistics(int_imgs)
assert s2["mean"].dtype == torch.float32

print("Problem 1: ALL TESTS PASSED")

---

## Problem 2: Label Counter and Class Weights

**Scenario:** You have an imbalanced video classification dataset (e.g., 90% "walking", 5% "jumping", 5% "dancing"). Before training, you need to compute inverse-frequency class weights so rare classes get higher loss weight.

**Spec:**

Given a list of integer labels, return a dict with:
- `"counts"` — dict mapping each class label (int) to its count
- `"frequencies"` — dict mapping each class label (int) to its frequency (count / total)
- `"weights"` — dict mapping each class label (int) to its inverse-frequency weight

**Weight formula:**

$$w_c = \frac{\text{total\_samples}}{\text{num\_classes} \times \text{count}_c}$$

This gives weight ≈ 1.0 for balanced classes, > 1.0 for rare classes, < 1.0 for common classes.

**Constraints:**
- Labels are non-negative integers (not necessarily contiguous).
- Return Python dicts with int keys and float values for frequencies and weights.

In [ ]:
def compute_class_weights(labels: list[int]) -> dict[str, dict[int, Any]]:
    """Compute class counts, frequencies, and inverse-frequency weights.

    Args:
        labels: List of integer class labels.

    Returns:
        Dict with keys 'counts', 'frequencies', 'weights'.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 2 ---

# Balanced case
labels_balanced = [0, 1, 2, 0, 1, 2]
result = compute_class_weights(labels_balanced)
assert set(result.keys()) == {"counts", "frequencies", "weights"}
assert result["counts"] == {0: 2, 1: 2, 2: 2}
assert abs(result["frequencies"][0] - 1/3) < 1e-7
assert abs(result["weights"][0] - 1.0) < 1e-7  # balanced -> weight = 1.0
assert abs(result["weights"][1] - 1.0) < 1e-7

# Imbalanced case
labels_imb = [0]*90 + [1]*5 + [2]*5
result2 = compute_class_weights(labels_imb)
assert result2["counts"][0] == 90
assert result2["counts"][1] == 5
assert abs(result2["frequencies"][0] - 0.9) < 1e-7
# weight_0 = 100 / (3 * 90) = 0.3703...
assert abs(result2["weights"][0] - 100 / 270) < 1e-7
# weight_1 = 100 / (3 * 5) = 6.6666...
assert abs(result2["weights"][1] - 100 / 15) < 1e-7

# Non-contiguous labels
labels_nc = [10, 10, 20, 20, 20, 50]
result3 = compute_class_weights(labels_nc)
assert set(result3["counts"].keys()) == {10, 20, 50}
assert result3["counts"][50] == 1
# weight_50 = 6 / (3 * 1) = 2.0
assert abs(result3["weights"][50] - 2.0) < 1e-7

# Single class
result4 = compute_class_weights([0, 0, 0])
assert result4["counts"] == {0: 3}
assert abs(result4["weights"][0] - 1.0) < 1e-7

print("Problem 2: ALL TESTS PASSED")

---

## Problem 3: Stratified Train/Test Split

**Scenario:** You're splitting a dataset for model validation. A naive random split could leave some rare classes entirely out of the test set. You need a stratified split that preserves class proportions.

**Spec:**

Given:
- `indices` — list of integer data indices (e.g., `[0, 1, 2, ..., N-1]`)
- `labels` — list of integer class labels, same length as `indices`
- `test_size` — float in `(0, 1)`, fraction of data for test set
- `seed` — int, random seed for reproducibility

Return `(train_indices, test_indices)` where:
- Each class contributes `floor(count_c * test_size)` samples to test (at least 1 if the class has enough samples and `floor > 0`).
- Remaining samples go to train.
- Within each class, samples are shuffled before splitting.
- Use `random.Random(seed)` for shuffling.

**Constraints:**
- Return two plain Python lists of ints.
- Every index appears in exactly one of train or test.

In [ ]:
def stratified_split(
    indices: list[int],
    labels: list[int],
    test_size: float,
    seed: int = 42,
) -> tuple[list[int], list[int]]:
    """Split data indices into train/test preserving class proportions.

    Args:
        indices: Data point indices.
        labels: Corresponding class labels.
        test_size: Fraction of data for the test set.
        seed: Random seed.

    Returns:
        (train_indices, test_indices)
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 3 ---

# Balanced case
indices = list(range(100))
labels = [0]*25 + [1]*25 + [2]*25 + [3]*25
train, test = stratified_split(indices, labels, test_size=0.2, seed=42)

# No overlap, full coverage
assert len(set(train) & set(test)) == 0
assert set(train) | set(test) == set(indices)

# Roughly 80/20 split
assert len(test) == 20  # 4 classes * floor(25*0.2) = 4*5 = 20
assert len(train) == 80

# Check class proportions in test set
test_labels = [labels[i] for i in test]
from collections import Counter
test_counts = Counter(test_labels)
assert test_counts[0] == 5
assert test_counts[1] == 5
assert test_counts[2] == 5
assert test_counts[3] == 5

# Reproducibility
train2, test2 = stratified_split(indices, labels, test_size=0.2, seed=42)
assert train == train2
assert test == test2

# Different seed gives different split
train3, test3 = stratified_split(indices, labels, test_size=0.2, seed=99)
assert test != test3  # very unlikely to be identical

# Imbalanced case
labels_imb = [0]*90 + [1]*10
indices_imb = list(range(100))
tr, te = stratified_split(indices_imb, labels_imb, test_size=0.2, seed=42)
te_labels = [labels_imb[i] for i in te]
te_counts = Counter(te_labels)
assert te_counts[0] == 18   # floor(90 * 0.2) = 18
assert te_counts[1] == 2    # floor(10 * 0.2) = 2

print("Problem 3: ALL TESTS PASSED")

---

## Problem 4: Min-Max and Z-Score Normalization

**Scenario:** Before feeding features into a model, you often normalize them. Two common approaches: min-max scaling to `[0, 1]` and z-score normalization (zero mean, unit variance).

**Spec:**

Implement two functions:

**`min_max_normalize(x, dim)`** — Scale values along `dim` to `[0, 1]`:

$$x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

If `max == min` along a slice, return zeros for that slice.

**`z_score_normalize(x, dim)`** — Standardize along `dim`:

$$x_{\text{norm}} = \frac{x - \mu}{\sigma}$$

If `std == 0` along a slice, return zeros for that slice.

**Constraints:**
- `dim` is a single integer dimension.
- Use `keepdim=True` for broadcasting.
- Return float32 tensors.

In [ ]:
def min_max_normalize(x: torch.Tensor, dim: int = 0) -> torch.Tensor:
    """Min-max normalize tensor to [0, 1] along given dimension.

    Args:
        x: Input tensor.
        dim: Dimension along which to normalize.

    Returns:
        Normalized tensor (same shape), float32.
    """
    # YOUR CODE HERE
    pass


def z_score_normalize(x: torch.Tensor, dim: int = 0) -> torch.Tensor:
    """Z-score normalize tensor along given dimension.

    Args:
        x: Input tensor.
        dim: Dimension along which to normalize.

    Returns:
        Normalized tensor (same shape), float32.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 4 ---

# Min-max basic
x = torch.tensor([[1.0, 2.0, 3.0],
                   [4.0, 5.0, 6.0]])
mm = min_max_normalize(x, dim=1)
assert mm.shape == x.shape
assert torch.allclose(mm[0], torch.tensor([0.0, 0.5, 1.0]))
assert torch.allclose(mm[1], torch.tensor([0.0, 0.5, 1.0]))

# Min-max along dim=0
mm0 = min_max_normalize(x, dim=0)
assert torch.allclose(mm0[0], torch.tensor([0.0, 0.0, 0.0]))
assert torch.allclose(mm0[1], torch.tensor([1.0, 1.0, 1.0]))

# Min-max edge case: constant row
const = torch.tensor([[3.0, 3.0, 3.0], [1.0, 2.0, 3.0]])
mm_const = min_max_normalize(const, dim=1)
assert torch.allclose(mm_const[0], torch.zeros(3))  # all same -> zeros

# Z-score basic
z = z_score_normalize(x, dim=1)
assert z.shape == x.shape
# Mean should be ~0 along normalized dim
assert torch.allclose(z.mean(dim=1), torch.zeros(2), atol=1e-5)
# Std should be ~1 along normalized dim
assert torch.allclose(z.std(dim=1), torch.ones(2), atol=1e-5)

# Z-score edge case: constant row
z_const = z_score_normalize(const, dim=1)
assert torch.allclose(z_const[0], torch.zeros(3))  # std=0 -> zeros

# dtype check
int_x = torch.tensor([[1, 5, 3]])
assert min_max_normalize(int_x, dim=1).dtype == torch.float32
assert z_score_normalize(int_x, dim=1).dtype == torch.float32

print("Problem 4: ALL TESTS PASSED")

---

## Problem 5: K-Means Clustering

**Scenario:** You're clustering video frame embeddings to discover visual themes in a dataset. Implement K-Means from scratch.

**Spec:**

Given:
- `data`: tensor of shape `(N, D)` — N data points, D dimensions
- `k`: int — number of clusters
- `max_iters`: int — maximum iterations (default 100)
- `seed`: int — random seed for reproducibility

Implement K-Means:
1. **Initialize centroids:** Use `torch.manual_seed(seed)`, then pick `k` random indices from `data` (use `torch.randperm(N)[:k]`) as initial centroids.
2. **Assign step:** For each point, find the nearest centroid by L2 distance.
3. **Update step:** Recompute each centroid as the mean of its assigned points.
4. **Convergence:** Stop if centroids don't change (within atol=1e-6) or max_iters reached.

Return a dict with:
- `"centroids"` — (k, D) tensor of final centroids
- `"assignments"` — (N,) int64 tensor of cluster indices
- `"iterations"` — int, number of iterations run
- `"inertia"` — float, sum of squared distances from each point to its assigned centroid

**Constraints:**
- No sklearn. Use tensor operations.
- Handle edge case: if a cluster loses all points, keep its centroid unchanged.

In [ ]:
def kmeans(data: torch.Tensor, k: int, max_iters: int = 100, seed: int = 42) -> dict:
    """K-Means clustering.

    Args:
        data: (N, D) tensor of data points.
        k: Number of clusters.
        max_iters: Maximum iterations.
        seed: Random seed.

    Returns:
        Dict with 'centroids', 'assignments', 'iterations', 'inertia'.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 5 ---
torch.manual_seed(42)

# Well-separated clusters
cluster_0 = torch.randn(50, 2) + torch.tensor([5.0, 5.0])
cluster_1 = torch.randn(50, 2) + torch.tensor([-5.0, -5.0])
cluster_2 = torch.randn(50, 2) + torch.tensor([5.0, -5.0])
data = torch.cat([cluster_0, cluster_1, cluster_2])

result = kmeans(data, k=3, seed=42)

# Check shapes
assert result["centroids"].shape == (3, 2), f"Expected (3,2), got {result['centroids'].shape}"
assert result["assignments"].shape == (150,), f"Expected (150,), got {result['assignments'].shape}"
assert result["assignments"].dtype == torch.int64

# Each cluster should have ~50 points
counts = torch.bincount(result["assignments"], minlength=3)
assert counts.min().item() >= 40, f"Clusters should be roughly balanced: {counts.tolist()}"

# Points from same original cluster should mostly share an assignment
labels_0 = result["assignments"][:50]
labels_1 = result["assignments"][50:100]
labels_2 = result["assignments"][100:]
# Majority label in each group should be consistent
majority_0 = labels_0.mode().values.item()
majority_1 = labels_1.mode().values.item()
majority_2 = labels_2.mode().values.item()
assert len({majority_0, majority_1, majority_2}) == 3, "Each original cluster should map to a different k-means cluster"
assert (labels_0 == majority_0).sum() >= 45, "Cluster 0 should be mostly pure"

# Inertia should be positive
assert result["inertia"] > 0

# Converges in reasonable iterations
assert result["iterations"] <= 20, f"Should converge quickly on well-separated data, took {result['iterations']}"

# Reproducibility
result2 = kmeans(data, k=3, seed=42)
assert torch.allclose(result["centroids"], result2["centroids"])

# k=1: everything in one cluster
result_1 = kmeans(data, k=1, seed=42)
assert result_1["centroids"].shape == (1, 2)
assert (result_1["assignments"] == 0).all()

print("Problem 5: ALL TESTS PASSED")

---

## Problem 6: KNN Classifier

**Scenario:** Quick baseline classifier for evaluating embedding quality — no training needed, just compare distances.

**Spec:**

Implement a K-Nearest Neighbors classifier:

**`KNNClassifier`:**
- `__init__(self, k: int = 5)` — number of neighbors
- `fit(self, X: torch.Tensor, y: torch.Tensor)` — store training data. X: (N, D), y: (N,) int64
- `predict(self, X_query: torch.Tensor) -> torch.Tensor` — predict labels for query points
  - For each query point, find the k nearest training points by L2 distance
  - Return the majority label among those k neighbors
  - Break ties by choosing the smallest label
  - Return (M,) int64 tensor
- `predict_with_distances(self, X_query: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]`
  - Return `(predictions, distances)` where distances is (M, k) tensor of distances to k nearest neighbors

**Distance formula:** `||a - b||_2` (Euclidean distance)

**Constraints:**
- Use vectorized pairwise distance computation (no loops over query points).
- No sklearn.

In [ ]:
class KNNClassifier:
    """K-Nearest Neighbors classifier.

    Args:
        k: Number of neighbors.
    """

    def __init__(self, k: int = 5):
        # YOUR CODE HERE
        pass

    def fit(self, X: torch.Tensor, y: torch.Tensor) -> None:
        # YOUR CODE HERE
        pass

    def predict(self, X_query: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass

    def predict_with_distances(self, X_query: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # YOUR CODE HERE
        pass

In [ ]:
# --- Tests: Problem 6 ---
torch.manual_seed(42)

# Simple 2D dataset
X_train = torch.tensor([
    [0.0, 0.0], [0.1, 0.1], [0.2, 0.0],   # class 0 (near origin)
    [5.0, 5.0], [5.1, 5.1], [4.9, 5.0],   # class 1 (far from origin)
    [10.0, 0.0], [10.1, 0.1], [9.9, 0.0], # class 2
])
y_train = torch.tensor([0, 0, 0, 1, 1, 1, 2, 2, 2])

knn = KNNClassifier(k=3)
knn.fit(X_train, y_train)

# Query points near each cluster
queries = torch.tensor([[0.05, 0.05], [5.0, 5.0], [10.0, 0.05]])
preds = knn.predict(queries)
assert preds.tolist() == [0, 1, 2], f"Expected [0,1,2], got {preds.tolist()}"
assert preds.dtype == torch.int64

# predict_with_distances
preds_d, dists = knn.predict_with_distances(queries)
assert preds_d.tolist() == [0, 1, 2]
assert dists.shape == (3, 3), f"Expected (3,3), got {dists.shape}"
assert (dists >= 0).all(), "Distances should be non-negative"
# Distances should be sorted ascending per query
for i in range(3):
    assert (dists[i, 1:] >= dists[i, :-1]).all(), f"Distances for query {i} should be sorted"

# k=1: nearest neighbor
knn1 = KNNClassifier(k=1)
knn1.fit(X_train, y_train)
preds1 = knn1.predict(X_train)  # predict on training data
assert (preds1 == y_train).all(), "k=1 on training data should give perfect accuracy"

# Tie-breaking: smallest label wins
X_tie = torch.tensor([[0.0, 0.0], [1.0, 0.0]])
y_tie = torch.tensor([1, 0])
knn_tie = KNNClassifier(k=2)
knn_tie.fit(X_tie, y_tie)
pred_tie = knn_tie.predict(torch.tensor([[0.5, 0.0]]))
assert pred_tie.item() == 0, f"Tie should break to smallest label, got {pred_tie.item()}"

# Batch prediction
X_batch = torch.randn(100, 10)
y_batch = torch.randint(0, 5, (100,))
knn_batch = KNNClassifier(k=5)
knn_batch.fit(X_batch, y_batch)
preds_batch = knn_batch.predict(torch.randn(20, 10))
assert preds_batch.shape == (20,)
assert preds_batch.dtype == torch.int64
assert (preds_batch >= 0).all() and (preds_batch < 5).all()

print("Problem 6: ALL TESTS PASSED")

---

## Problem 7: Linear Regression (Closed-Form + Gradient Descent)

**Scenario:** Fundamental ML algorithm — predict a continuous target from features. Implement both the analytical solution and iterative optimization.

**Spec:**

**Closed-form (Normal Equation):**

$$\mathbf{w} = (X^T X)^{-1} X^T y$$

Add a bias column of ones to X before solving.

**Gradient Descent:**

$$\hat{y} = X w$$
$$\text{loss} = \frac{1}{N} \|y - \hat{y}\|^2$$
$$\nabla_w = -\frac{2}{N} X^T (y - \hat{y})$$
$$w \leftarrow w - \eta \nabla_w$$

**`linear_regression_closed(X, y)`** — return weight vector w of shape (D+1,) where last element is bias
**`linear_regression_gd(X, y, lr, n_iters)`** — return (w, losses) where losses is list of MSE at each iteration

**Constraints:**
- No sklearn, no autograd. Pure tensor operations.
- X: (N, D), y: (N,)

In [ ]:
def linear_regression_closed(X: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """Closed-form linear regression via normal equation.

    Args:
        X: (N, D) feature matrix.
        y: (N,) target vector.

    Returns:
        (D+1,) weight vector (last element is bias).
    """
    # YOUR CODE HERE
    pass


def linear_regression_gd(
    X: torch.Tensor, y: torch.Tensor, lr: float = 0.01, n_iters: int = 1000
) -> tuple[torch.Tensor, list[float]]:
    """Linear regression via gradient descent.

    Args:
        X: (N, D) feature matrix.
        y: (N,) target vector.
        lr: Learning rate.
        n_iters: Number of iterations.

    Returns:
        ((D+1,) weight vector, list of MSE losses per iteration).
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 7 ---
torch.manual_seed(42)

# Simple linear relationship: y = 2*x1 + 3*x2 + 1
N = 200
X = torch.randn(N, 2)
true_w = torch.tensor([2.0, 3.0])
true_b = 1.0
y = X @ true_w + true_b + 0.01 * torch.randn(N)  # small noise

# Closed form
w_closed = linear_regression_closed(X, y)
assert w_closed.shape == (3,), f"Expected (3,), got {w_closed.shape}"
assert abs(w_closed[0].item() - 2.0) < 0.1, f"w1 should be ~2.0, got {w_closed[0].item():.3f}"
assert abs(w_closed[1].item() - 3.0) < 0.1, f"w2 should be ~3.0, got {w_closed[1].item():.3f}"
assert abs(w_closed[2].item() - 1.0) < 0.1, f"bias should be ~1.0, got {w_closed[2].item():.3f}"

# Predictions should be accurate
X_aug = torch.cat([X, torch.ones(N, 1)], dim=1)
preds_closed = X_aug @ w_closed
mse_closed = ((preds_closed - y) ** 2).mean().item()
assert mse_closed < 0.01, f"Closed-form MSE should be tiny, got {mse_closed:.4f}"

# Gradient descent
w_gd, losses = linear_regression_gd(X, y, lr=0.01, n_iters=500)
assert w_gd.shape == (3,)
assert len(losses) == 500

# Loss should decrease
assert losses[-1] < losses[0], "Loss should decrease over iterations"
assert losses[-1] < 0.1, f"Final loss should be small, got {losses[-1]:.4f}"

# GD weights should be close to closed-form (given enough iterations)
assert abs(w_gd[0].item() - w_closed[0].item()) < 0.5, "GD w1 should approach closed-form"

# 1D case
X_1d = torch.randn(100, 1)
y_1d = 3.0 * X_1d.squeeze() + 2.0
w_1d = linear_regression_closed(X_1d, y_1d)
assert w_1d.shape == (2,)
assert abs(w_1d[0].item() - 3.0) < 0.1

print("Problem 7: ALL TESTS PASSED")

---

## Problem 8: Logistic Regression

**Scenario:** Binary classification baseline — implement sigmoid, binary cross-entropy, and gradient descent training from scratch.

**Spec:**

**Sigmoid:** $\sigma(z) = \frac{1}{1 + e^{-z}}$

**Binary Cross-Entropy Loss:**
$$L = -\frac{1}{N} \sum_{i=1}^{N} [y_i \log(\hat{y}_i) + (1-y_i) \log(1 - \hat{y}_i)]$$

Clamp predictions to `[eps, 1-eps]` where `eps=1e-7` to avoid log(0).

**Gradient:**
$$\hat{y} = \sigma(Xw + b)$$
$$\nabla_w = \frac{1}{N} X^T (\hat{y} - y)$$
$$\nabla_b = \frac{1}{N} \sum(\hat{y} - y)$$

**`LogisticRegression`:**
- `__init__(self, input_dim: int, lr: float = 0.1)`
- `self.w`: (D,) weights initialized to zeros
- `self.b`: scalar bias initialized to zero
- `predict_proba(self, X)` — return (N,) probabilities
- `predict(self, X, threshold=0.5)` — return (N,) int64 binary predictions
- `train_step(self, X, y)` — one GD step, return loss (float)
- `fit(self, X, y, n_iters=1000)` — train for n_iters, return list of losses

**Constraints:**
- No autograd, no sklearn. Pure tensor math.

In [ ]:
class LogisticRegression:
    """Binary logistic regression classifier.

    Args:
        input_dim: Number of input features.
        lr: Learning rate.
    """

    def __init__(self, input_dim: int, lr: float = 0.1):
        # YOUR CODE HERE
        pass

    def predict_proba(self, X: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass

    def predict(self, X: torch.Tensor, threshold: float = 0.5) -> torch.Tensor:
        # YOUR CODE HERE
        pass

    def train_step(self, X: torch.Tensor, y: torch.Tensor) -> float:
        # YOUR CODE HERE
        pass

    def fit(self, X: torch.Tensor, y: torch.Tensor, n_iters: int = 1000) -> list[float]:
        # YOUR CODE HERE
        pass

In [ ]:
# --- Tests: Problem 8 ---
torch.manual_seed(42)

# Linearly separable data
N = 200
X = torch.randn(N, 2)
true_boundary = X[:, 0] + X[:, 1]  # decision boundary: x1 + x2 = 0
y = (true_boundary > 0).float()

model = LogisticRegression(input_dim=2, lr=0.5)

# Before training, predict_proba should return ~0.5 (zero weights)
probs_init = model.predict_proba(X)
assert probs_init.shape == (N,)
assert abs(probs_init.mean().item() - 0.5) < 1e-5, "Initial probs should be 0.5"

# Train
losses = model.fit(X, y, n_iters=500)
assert len(losses) == 500
assert losses[-1] < losses[0], "Loss should decrease"
assert losses[-1] < 0.3, f"Final loss should be low, got {losses[-1]:.4f}"

# Predictions should be accurate
preds = model.predict(X)
assert preds.dtype == torch.int64
accuracy = (preds == y.long()).float().mean().item()
assert accuracy > 0.90, f"Accuracy should be > 90%, got {accuracy:.1%}"

# Probabilities should be in [0, 1]
probs = model.predict_proba(X)
assert (probs >= 0).all() and (probs <= 1).all()

# Threshold works
preds_strict = model.predict(X, threshold=0.9)
assert preds_strict.sum() <= preds.sum(), "Higher threshold should predict fewer positives"

# Single train_step returns loss
loss_val = model.train_step(X, y)
assert isinstance(loss_val, float)
assert loss_val > 0

# 1D case
X_1d = torch.randn(100, 1)
y_1d = (X_1d.squeeze() > 0).float()
model_1d = LogisticRegression(input_dim=1, lr=1.0)
losses_1d = model_1d.fit(X_1d, y_1d, n_iters=300)
acc_1d = (model_1d.predict(X_1d) == y_1d.long()).float().mean().item()
assert acc_1d > 0.85, f"1D accuracy should be > 85%, got {acc_1d:.1%}"

print("Problem 8: ALL TESTS PASSED")

---

## Scoring Yourself

| Result | Assessment |
|--------|------------|
| 8/8 in < 20 min each | Ready for MEDIUM tier |
| 6-7/8 | Close — review the ones you missed, redo tomorrow |
| 4-5/8 | Gaps in fundamentals — drill these patterns daily |
| < 4/8 | Spend a week on basics before exam prep |